In [1]:
import os
import sys

sys.stderr = open(os.devnull, "w")

import warnings
import multiprocessing as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from datetime import timedelta
from dateutil.relativedelta import relativedelta
from sktime.forecasting.tbats import TBATS
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.layers import Input, Dense, LSTM, Dropout

warnings.filterwarnings('ignore', category=FutureWarning)

tf.random.set_seed(42)
np.random.seed(42)

/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/jchan/miniconda3/envs/sales_forecasting/lib/python3.12/site-packages/

In [2]:
sales = pd.read_csv('pedidos.csv', sep=';')

In [3]:
df_count = sales[['account_id', 'sales_channel_id']].value_counts()
df_count = pd.DataFrame(df_count).reset_index()

In [4]:
EPOCHS = 1000
BATCH_SIZE = 32
VALID_SPLIT = 0.1

In [5]:
ALL_SALES_CHANNELS = True
LOOKBACK = 3
CRIT_VALUE = 0.25
COVERAGE = 0.2
END_DATE = pd.to_datetime(pd.to_datetime(sales['created_date'].max()).strftime("%Y-%m-%d %H:00:00"))
START_DATE = END_DATE - timedelta(hours=END_DATE.hour)
TEST_SIZE = int((END_DATE - START_DATE).seconds / 60**2 + 1)

In [6]:
OOT_DATE = START_DATE + timedelta(hours=23)

In [7]:
OOT_PERIODS = int((OOT_DATE - END_DATE).seconds / 60**2 + 1)

In [8]:
oot_dates = pd.DatetimeIndex([END_DATE+timedelta(hours=h) for h in range(1, OOT_PERIODS)], freq='h')
df_oot = pd.DataFrame(index=oot_dates)

In [9]:
def preprocess_sales_data(account_id, sales_channel_id):
    if sales_channel_id == 'ALL':
        cond = (sales['account_id'] == account_id) & \
               (sales['status'].notna())
    else:
        cond = (sales['account_id'] == account_id) & \
               (sales['sales_channel_id'] == sales_channel_id) & \
               (sales['status'].notna())
    df_client = sales[cond].drop(['account_id', 'sales_channel_id'], axis=1)
    df_client['created_date'] = pd.to_datetime(df_client['created_date'], format='%Y-%m-%d %H:%M:%S.%f %z')
    df_client = df_client.sort_values('created_date').reset_index(drop=True)
    df_client['created_date'] = df_client['created_date'].dt.strftime("%Y-%m-%d %H:00:00").reset_index(drop=True)

    df_client_mod = df_client.groupby('created_date').agg(
        price_total_agg=('price_total', 'sum'),
        n_orders=('created_date', 'count')
    )
    df_client_mod.index = pd.to_datetime(df_client_mod.index, format='%Y-%m-%d %H:00:00')

    end_date = pd.to_datetime(END_DATE, format='%Y-%m-%d %H:%M:%S')
    start_date = end_date - relativedelta(months=LOOKBACK)
    date_index = pd.Series(pd.date_range(start=start_date, end=end_date, freq='h', name='created_date'))
    df_client_mod = pd.merge(date_index, df_client_mod, how='left', on='created_date').set_index('created_date')

    oot_date_index = pd.Series(df_oot.index, name='created_date')
    combined_date_index = pd.concat([date_index, oot_date_index], axis=0)
    df_client_mod = pd.merge(combined_date_index, df_client_mod, how='left', on='created_date').set_index('created_date')

    return df_client_mod


def create_lagged_features(df, lags=24):
    lagged_features = [df.shift(l).rename(f'lag_{l}') for l in range(1, lags + 1)]
    df_lag = pd.concat([df, pd.concat(lagged_features, axis=1)], axis=1).dropna().asfreq('h')
    return df_lag


def split_data(df_lag):
    df_train = df_lag.loc[df_lag.index < START_DATE].copy()
    df_test = df_lag.loc[(df_lag.index >= START_DATE) & (df_lag.index <= END_DATE)].copy()
    df_oot = df_lag.loc[df_lag.index > END_DATE].copy()
    return df_train, df_test, df_oot


def train_and_forecast(df_train, df_test, df_full, df_oot, col):
    X_train, y_train = df_train.drop(col, axis=1), df_train[col]
    X_test = df_test.drop(col, axis=1)
    X_full, y_full = df_full.drop(col, axis=1), df_full[col]
    X_oot = df_oot.drop(col, axis=1)
    
    train_mad = y_train.std()
    mad = y_full.std()
    
    tbats = TBATS(sp=[24, 168], show_warnings=False, n_jobs=-1)
    tbats.fit(y_train)
    tbats_estimations = tbats.predict(fh=X_train.index)
    
    gscv = GridSearchCV(GradientBoostingRegressor(loss="quantile", alpha=0.5, random_state=42, n_iter_no_change=3),
                        param_grid={'max_depth': [1, 2, 4, 8, 16],
                                    'subsample': [0.2, 0.4, 0.6, 0.8, 1],
                                    'max_features': ['sqrt', 'log2', None]},
                        scoring='neg_mean_absolute_error',
                        cv=TimeSeriesSplit(),
                        n_jobs=-1,
                        verbose=1)
    gscv.fit(X_train, y_train)
    gb = gscv.best_estimator_
    gb_estimations = pd.Series(gb.predict(X_train), index=X_train.index)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    X_train_scaled = np.reshape(X_train_scaled, (X_train.shape[0], 1, X_train.shape[1]))
    X_test_scaled = np.reshape(X_test_scaled, (X_test.shape[0], 1, X_test.shape[1]))
    
    early_stopping = EarlyStopping(patience=3)
    lstm = Sequential()
    lstm.add(Input(shape=(1, 24)))
    lstm.add(LSTM(128, activation='relu', return_sequences=True))
    lstm.add(Dropout(0.5))
    lstm.add(Dense(1))
    lstm.compile(loss='mean_absolute_error', optimizer='adam')
    
    lstm.fit(X_train_scaled, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=VALID_SPLIT, shuffle=False, callbacks=early_stopping, verbose=2)
    lstm_estimations = pd.Series(lstm.predict(X_train_scaled).squeeze(), index=X_train.index)
    
    models_estimations = pd.concat([tbats_estimations, gb_estimations, lstm_estimations], axis=1)
    models_estimations.columns = ['tbats', 'gb', 'lstm']
    model = LinearRegression()
    model.fit(models_estimations, y_train)
    estimations = pd.Series(model.predict(models_estimations), index=X_train.index)

    tbats_forecast = tbats.predict(fh=X_test.index)
    tbats_forecast_int = tbats.predict_interval(fh=X_test.index, coverage=COVERAGE)
    tbats_forecast_lower = tbats_forecast_int.iloc[:, 0]
    tbats_forecast_upper = tbats_forecast_int.iloc[:, 1]
    
    gb_forecast = pd.Series(gb.predict(X_test), index=X_test.index)
    gb_forecast_lower = gb_forecast - 0.25*train_mad
    gb_forecast_upper = gb_forecast + 0.25*train_mad
    
    lstm_forecast = pd.Series(lstm.predict(X_test_scaled).squeeze(), index=X_test.index)
    lstm_forecast_lower = lstm_forecast - 0.25*train_mad
    lstm_forecast_upper = lstm_forecast + 0.25*train_mad
    
    models_forecast = pd.concat([tbats_forecast, gb_forecast, lstm_forecast], axis=1)
    models_forecast.columns = ['tbats', 'gb', 'lstm']
    
    models_forecast_lower = pd.concat([tbats_forecast_lower, gb_forecast_lower, lstm_forecast_lower], axis=1)
    models_forecast_upper = pd.concat([tbats_forecast_upper, gb_forecast_upper, lstm_forecast_upper], axis=1)
    
    forecast = pd.Series(model.predict(models_forecast), index=X_test.index)
    forecast_lower = pd.Series(model.predict(models_forecast_lower), index=X_test.index)
    forecast_upper = pd.Series(model.predict(models_forecast_upper), index=X_test.index)
    
    oot_tbats = TBATS(sp=[24, 168], show_warnings=False, n_jobs=-1)
    oot_tbats.fit(y_full)
    
    oot_tbats_forecast = oot_tbats.predict(fh=X_oot.index)
    oot_tbats_forecast_int = oot_tbats.predict_interval(fh=X_oot.index, coverage=COVERAGE)
    oot_tbats_forecast_lower = oot_tbats_forecast_int.iloc[:, 0]
    oot_tbats_forecast_upper = oot_tbats_forecast_int.iloc[:, 1]
    
    oot_gscv = GridSearchCV(GradientBoostingRegressor(loss="quantile", alpha=0.5, random_state=42, n_iter_no_change=3),
                            param_grid={'max_depth': [1, 2, 4, 8, 16],
                                        'subsample': [0.2, 0.4, 0.6, 0.8, 1],
                                        'max_features': ['sqrt', 'log2', None]},
                            scoring='neg_mean_absolute_error',
                            cv=TimeSeriesSplit(),
                            n_jobs=-1,
                            verbose=1)        
    oot_gscv.fit(X_full, y_full)
    oot_gb = oot_gscv.best_estimator_
    
    scaler = StandardScaler()
    X_full_scaled = np.reshape(scaler.fit_transform(X_full), (X_full.shape[0], 1, X_full.shape[1]))
    
    early_stopping = EarlyStopping(patience=3)
    oot_lstm = Sequential()
    oot_lstm.add(Input(shape=(1, 24)))
    oot_lstm.add(LSTM(128, activation='relu', return_sequences=True))
    oot_lstm.add(Dropout(0.5))
    oot_lstm.add(Dense(1))
    oot_lstm.compile(loss='mean_absolute_error', optimizer='adam')
    oot_lstm.fit(X_full_scaled, y_full, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=VALID_SPLIT, shuffle=False, callbacks=early_stopping, verbose=2)
    
    oot_tbats_estimations = pd.Series(oot_tbats.predict(fh=X_full.index), index=X_full.index)
    oot_gb_estimations = pd.Series(oot_gb.predict(X_full), index=X_full.index)
    oot_lstm_estimations = pd.Series(oot_lstm.predict(X_full_scaled).squeeze(), index=X_full.index)
    
    oot_estimations = pd.concat([oot_tbats_estimations, oot_gb_estimations, oot_lstm_estimations], axis=1)
    oot_estimations.columns = ['tbats', 'gb', 'lstm']
    oot_model = LinearRegression()
    oot_model.fit(oot_estimations, y_full)
    
    display(pd.DataFrame(zip(oot_model.feature_names_in_, oot_model.coef_), columns=['feature', 'coef']))
    
    oot_gb_forecast_ls = []
    oot_gb_forecast_lower_ls = []
    oot_gb_forecast_upper_ls = []
    oot_lstm_forecast_ls = []
    oot_lstm_forecast_lower_ls = []
    oot_lstm_forecast_upper_ls = []
    
    X_oot_gb = X_oot.copy()
    X_oot_lstm = X_oot.copy()
    for i in range(len(X_oot)):
        x_oot_gb = X_oot_gb.iloc[i].to_frame().T
        
        x_oot_lstm = X_oot_lstm.iloc[i].to_frame().T
        x_oot_lstm_scaled = scaler.transform(x_oot_lstm)
        x_oot_lstm_scaled = np.reshape(x_oot_lstm_scaled, (x_oot_lstm.shape[0], 1, x_oot_lstm.shape[1]))
        
        x_oot_gb_forecast = oot_gb.predict(x_oot_gb)[0]
        oot_gb_forecast_ls.append(x_oot_gb_forecast)
        oot_gb_forecast_lower_ls.append(x_oot_gb_forecast - CRIT_VALUE*mad)
        oot_gb_forecast_upper_ls.append(x_oot_gb_forecast + CRIT_VALUE*mad)
        
        x_oot_lstm_forecast = oot_lstm.predict(x_oot_lstm_scaled).squeeze()
        oot_lstm_forecast_ls.append(x_oot_lstm_forecast)
        oot_lstm_forecast_lower_ls.append(x_oot_lstm_forecast - CRIT_VALUE*mad)
        oot_lstm_forecast_upper_ls.append(x_oot_lstm_forecast + CRIT_VALUE*mad)
        
        for j, k in list(zip(range(i+1, len(X_oot)), range(len(X_oot)-i-1))):
            X_oot_gb.iloc[j, k] = x_oot_gb_forecast
            X_oot_lstm.iloc[j, k] = x_oot_lstm_forecast
    
    oot_gb_forecast = pd.Series(oot_gb_forecast_ls, index=X_oot.index)
    oot_gb_forecast_lower = pd.Series(oot_gb_forecast_lower_ls, index=X_oot.index)
    oot_gb_forecast_upper = pd.Series(oot_gb_forecast_upper_ls, index=X_oot.index)
    
    print(oot_gb_forecast_lower)
    print(oot_gb_forecast_upper)
    
    oot_lstm_forecast = pd.Series(oot_lstm_forecast_ls, index=X_oot.index)
    oot_lstm_forecast_lower = pd.Series(oot_lstm_forecast_lower_ls, index=X_oot.index)
    oot_lstm_forecast_upper = pd.Series(oot_lstm_forecast_upper_ls, index=X_oot.index)
    
    oot_models_forecast = pd.concat([oot_tbats_forecast, oot_gb_forecast, oot_lstm_forecast], axis=1)
    oot_models_forecast.columns = ['tbats', 'gb', 'lstm']
    
    display(oot_models_forecast)
    
    oot_models_forecast_lower = pd.concat([oot_tbats_forecast_lower, oot_gb_forecast_lower, oot_lstm_forecast_lower], axis=1)
    oot_models_forecast_upper = pd.concat([oot_tbats_forecast_upper, oot_gb_forecast_upper, oot_lstm_forecast_upper], axis=1)
    
    oot_forecast = pd.Series(oot_model.predict(oot_models_forecast), index=X_oot.index)
    oot_forecast_lower = pd.Series(oot_model.predict(oot_models_forecast_lower), index=X_oot.index)
    oot_forecast_upper = pd.Series(oot_model.predict(oot_models_forecast_upper), index=X_oot.index)

    return {
        'estimations': estimations,
        'forecast': forecast,
        'forecast_lower': forecast_lower,
        'forecast_upper': forecast_upper,
        'oot_forecast': oot_forecast,
        'oot_forecast_lower': oot_forecast_lower,
        'oot_forecast_upper': oot_forecast_upper
    }
    
    
def compute_metrics_and_plot_time_series(forecast_results, df_train, df_test, col):
    y_train = df_train[col]
    y_test = df_test[col]
    estimations = forecast_results['estimations']
    forecast = forecast_results['forecast']
    forecast_lower = forecast_results['forecast_lower']
    forecast_upper = forecast_results['forecast_upper']
    oot_forecast = forecast_results['oot_forecast']
    oot_forecast_lower = forecast_results['oot_forecast_lower']
    oot_forecast_upper = forecast_results['oot_forecast_upper']
    
    train_mae = mean_absolute_error(df_train[col], estimations)
    train_rmse = root_mean_squared_error(df_train[col], estimations)
    print(f"train RMSE: {train_rmse:.3f}")
    print(f"train MAE:  {train_mae:.3f}")
    test_mae = mean_absolute_error(df_test[col], forecast)
    test_rmse = root_mean_squared_error(df_test[col], forecast)
    print(f"test RMSE: {test_rmse:.3f}")
    print(f"test MAE:  {test_mae:.3f}")
    
    plt.figure(figsize=(12, 3))
    sns.lineplot(y_train.iloc[-5*TEST_SIZE:], lw=0.8, label='Train')
    sns.lineplot(estimations.iloc[-5*TEST_SIZE:], lw=0.8, label='Estimation')
    sns.lineplot(y_test, lw=0.8, label='Test')
    sns.lineplot(forecast, lw=0.8, label='Forecast')
    sns.lineplot(forecast_lower, c='darkviolet', linestyle='--', lw=0.8)
    sns.lineplot(forecast_upper, c='darkviolet', linestyle='--', lw=0.8)
    sns.lineplot(2*forecast_lower-forecast, c='darkviolet', alpha=0.3, linestyle='--', lw=0.8)
    sns.lineplot(2*forecast_upper-forecast, c='darkviolet', alpha=0.3, linestyle='--', lw=0.8)
    sns.lineplot(oot_forecast, lw=0.8, label='OOT Forecast')
    sns.lineplot(oot_forecast_lower, c='darkviolet', linestyle='--', lw=0.8)
    sns.lineplot(oot_forecast_upper, c='darkviolet', linestyle='--', lw=0.8)
    plt.fill_between(x=forecast.index,
                     y1=forecast_lower,
                     y2=forecast_upper,
                     color='violet',
                     alpha=0.5)
    plt.fill_between(x=forecast.index,
                     y1=2*forecast_lower-forecast,
                     y2=2*forecast_upper-forecast,
                     color='violet',
                     alpha=0.3)
    plt.fill_between(x=oot_forecast.index,
                     y1=oot_forecast_lower,
                     y2=oot_forecast_upper,
                     color='violet',
                     alpha=0.3)
    plt.legend()
    plt.show()
    
    return train_rmse, test_rmse, train_mae, test_mae


def save_forecast_to_sql(account_id, sales_channel_id, df_test, df_oot, forecast_dict):
    with open(f'sql/time_series/forecast_{account_id}_{sales_channel_id}_ts.sql', 'w') as output_file:
        for i in range(len(df_test)):
            start_time = df_test.index[i]
            end_time = start_time + pd.Timedelta(hours=1)
            output_file.write(
                f"""
                INSERT INTO forecast (id, created, modified, platform, store_name, "start", "end",
                                      channel, seller, account_id, organization_id, store_id, minutes_interval,
                                      model, orders_high, orders_low, orders_mean, sales_high, sales_low, sales_mean)
                VALUES (gen_random_uuid(), now(), now(), 1, (select vtexid from vtex_account where id = '{account_id}'::uuid), '{start_time}', '{end_time}',
                        {sales_channel_id}, 'ALL', '{account_id}'::uuid, (select organizationid from vtex_account where id = '{account_id}'::uuid), NULL, 60, 'TBATS',
                        {forecast_dict['dataset_1']['forecast_upper'].iloc[i]}, {forecast_dict['dataset_1']['forecast_lower'].iloc[i]}, {forecast_dict['dataset_1']['forecast'].iloc[i]},
                        {forecast_dict['dataset_0']['forecast_upper'].iloc[i]}, {forecast_dict['dataset_0']['forecast_lower'].iloc[i]}, {forecast_dict['dataset_0']['forecast'].iloc[i]});
                """
            )
        for i in range(len(df_oot)):
            start_time = df_oot.index[i]
            end_time = start_time + pd.Timedelta(hours=1)
            output_file.write(
                f"""
                INSERT INTO forecast (id, created, modified, platform, store_name, "start", "end",
                                      channel, seller, account_id, organization_id, store_id, minutes_interval,
                                      model, orders_high, orders_low, orders_mean, sales_high, sales_low, sales_mean)
                VALUES (gen_random_uuid(), now(), now(), 1, (select vtexid from vtex_account where id = '{account_id}'::uuid), '{start_time}', '{end_time}',
                        {sales_channel_id}, 'ALL', '{account_id}'::uuid, (select organizationid from vtex_account where id = '{account_id}'::uuid), NULL, 60, 'TBATS',
                        {forecast_dict['dataset_1']['oot_forecast_upper'].iloc[i]}, {forecast_dict['dataset_1']['oot_forecast_lower'].iloc[i]}, {forecast_dict['dataset_1']['oot_forecast'].iloc[i]},
                        {forecast_dict['dataset_0']['oot_forecast_upper'].iloc[i]}, {forecast_dict['dataset_0']['oot_forecast_lower'].iloc[i]}, {forecast_dict['dataset_0']['oot_forecast'].iloc[i]});
                """
            )


# Main
if ALL_SALES_CHANNELS:
    id_pairs = set(list(zip(df_count['account_id'], ['ALL']*len(df_count))))
else:
    id_pairs = list(zip(df_count['account_id'], df_count['sales_channel_id']))

summary_dict = {}
for account_id, sales_channel_id in id_pairs:
    print(f'Processing account_id: {account_id}, sales_channel_id: {sales_channel_id}')
    summary_dict[(account_id, sales_channel_id)] = {}

    df_client_mod = preprocess_sales_data(account_id, sales_channel_id)
    df_client_mod.fillna(0, inplace=True)

    forecast_dict = {}
    for n, col in enumerate(['price_total_agg', 'n_orders']):
        print(f'Processing column: {col}')
        summary_dict[(account_id, sales_channel_id)][f'dataset_{n}'] = {'name': col}

        df_lag = create_lagged_features(df_client_mod[col])
        df_train, df_test, df_oot = split_data(df_lag)
        df_full = pd.concat([df_train, df_test, df_oot], axis=0)

        forecast_results = train_and_forecast(df_train, df_test, df_full, df_oot, col)
        
        forecast_dict[f'dataset_{n}'] = {}
        forecast_dict[f'dataset_{n}'].update(forecast_results)
        
        train_rmse, test_rmse, train_mae, test_mae = compute_metrics_and_plot_time_series(forecast_results, df_train, df_test, col)
        summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['train RMSE'] = train_rmse
        summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['test RMSE'] = test_rmse
        summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['train MAE'] = train_mae
        summary_dict[(account_id, sales_channel_id)][f'dataset_{n}']['test MAE'] = test_mae

    save_forecast_to_sql(account_id, sales_channel_id, df_test, df_oot, forecast_dict)

Processing account_id: 60ca8452-5b14-481f-a637-8756f527aee8, sales_channel_id: ALL
Processing column: price_total_agg
Fitting 5 folds for each of 75 candidates, totalling 375 fits
Epoch 1/1000
60/60 - 1s - 24ms/step - loss: 275.6743 - val_loss: 0.0031
Epoch 2/1000
60/60 - 0s - 4ms/step - loss: 275.6263 - val_loss: 0.0012
Epoch 3/1000
60/60 - 0s - 3ms/step - loss: 275.5316 - val_loss: 3.2372e-04
Epoch 4/1000
60/60 - 0s - 3ms/step - loss: 275.3454 - val_loss: 1.8694e-05
Epoch 5/1000
60/60 - 0s - 3ms/step - loss: 275.0287 - val_loss: 3.2633e-05
Epoch 6/1000
60/60 - 0s - 3ms/step - loss: 274.6204 - val_loss: 1.1918e-04
Epoch 7/1000
60/60 - 0s - 3ms/step - loss: 274.1822 - val_loss: 2.6169e-04
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Fitting 5 folds for each of 75 candidates, totalling 375 fits
Epoch 1/1000
61/61 - 1s - 12ms/step - loss: 272.5374 - val_loss: 0.0010
Epoch 2/1000
61/61 - 0s - 3ms/step - loss: 272.4925 - val_loss: 3.8673e-04
Epoch 3/1000
61/

,feature,coef
0,tbats,0.991435
1,gb,0.000000
2,lstm,-0.640527


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
created_date
2025-03-27 11:00:00   -331.268991
2025-03-27 12:00:00   -331.268991
2025-03-27 13:00:00   -331.268991
2025-03-27 14:00:00   -331.268991
2025-03-27 15:00:00   -331.268991
2025-03-27 16:00:00   -331.268991
2025-03-27 17:00:00   -331.268991
2025-03-27 18:00:00   -331.268991
2025-03-27 19:00:00   -331.268991
2025-03-27 20:00:00   -331.268991
2025-03-27 21:00:00   -331.268991
2025-03-27 22:00:00   -331.268991
2025-03-27 23:00:00   -331.268991
Freq: h, dtype: float64
created_date
2025-03-27 11:

,tbats,gb,lstm
created_date,,,
2025-03-27 11:00:00,0.000051,0.0,-0.00016137007
2025-03-27 12:00:00,0.000050,0.0,-0.00016137007
2025-03-27 13:00:00,0.000050,0.0,-0.00016137007
2025-03-27 14:00:00,0.000049,0.0,-0.0001613701
2025-03-27 15:00:00,0.000049,0.0,-0.00016137009
2025-03-27 16:00:00,0.000048,0.0,-0.00016137009
2025-03-27 17:00:00,0.000048,0.0,-0.0001613701
2025-03-27 18:00:00,0.000047,0.0,-0.00016137009
2025-03-27 19:00:00,0.000047,0.0,-0.00016137009


ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
dataset_0 = pd.DataFrame({k: v['dataset_0'] for k, v in summary_dict.items()}).T
dataset_1 = pd.DataFrame({k: v['dataset_1'] for k, v in summary_dict.items()}).T

In [ ]:
summary = (
    pd
    .concat([dataset_0, dataset_1], axis=0)
    .rename({'name': 'dataset'}, axis=1)
    .sort_index()
    .reset_index(names=['account_id', 'sales_channel_id'])
)
summary.to_csv('summary_ml.csv', sep=';')

summary

In [ ]:
train_metrics = summary.groupby('dataset')['train MAE'].mean()

train_metrics

In [ ]:
test_metrics = summary.groupby('dataset')['test MAE'].mean()

test_metrics

In [ ]:
(train_metrics - test_metrics).abs()